In [9]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


print("\n--- NGÀY 94: MẢNH GHÉP CUỐI CÙNG CỦA RAG (GENERATION) ---")

load_dotenv()
os.environ["GOOGLE_API_KEY"] = os.getenv("GEMINI_API_KEY")

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.7
)

may_ma_hoa = GoogleGenerativeAIEmbeddings(
     model="models/gemini-embedding-001"
)

#Kêt nối DB
print("[1] Đang đánh thức bộ nhớ ChromaDB...")
vector_db = Chroma(
    persist_directory="F:/Neuro-sama/Jarvis_Project/notebooks/Phase2/database",
    embedding_function=may_ma_hoa
)

nguoi_tim_kiem = vector_db.as_retriever(search_kwargs={"k":3})

#Trong LangChain biến {context} là từ khóa bắt buộc
mau_lenh="""
Bạn là một trợ lý AI thông minh. Hãy trả lời câu hỏi dựa TRÊN TÀI LIỆU được cung cấp dưới đây.
Nếu tài liệu không chứa câu trả lời, hãy nói "Tôi không biết dựa trên tài liệu hiện tại", tuyệt đối KHÔNG tự bịa ra thông tin.

TÀI LIỆU:
{context}

CÂU HỎI: {input}
"""

prompt=ChatPromptTemplate.from_template(mau_lenh)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

print("[2] Đang kích hoạt dây chuyền tư duy LCEL...")

day_chuyen_rag= (
    {'context':nguoi_tim_kiem | format_docs, 'input': RunnablePassthrough()}
    |prompt
    |llm
    |StrOutputParser()
)

cau_hoi = "what is agent"
print(f"\n[USER]: {cau_hoi}")
print("[HỆ THỐNG]: Đang tự động tìm kiếm tài liệu và đọc suy nghĩ...\n")

# Lệnh invoke sẽ tự động kích hoạt cả quá trình: Tìm kiếm -> Nhét Prompt -> Gọi LLM
ket_qua = day_chuyen_rag.invoke(cau_hoi)

print("=> [JARVIS]:")
print("--------------------------------------------------")
# LangChain trả về kết quả nằm trong key 'answer'
print(ket_qua)
print("--------------------------------------------------")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.



--- NGÀY 94: MẢNH GHÉP CUỐI CÙNG CỦA RAG (GENERATION) ---
[1] Đang đánh thức bộ nhớ ChromaDB...
[2] Đang kích hoạt dây chuyền tư duy LCEL...

[USER]: what is agent
[HỆ THỐNG]: Đang tự động tìm kiếm tài liệu và đọc suy nghĩ...

=> [JARVIS]:
--------------------------------------------------
Tôi không biết dựa trên tài liệu hiện tại
--------------------------------------------------
